# Structured streaming Demo

Consumer PySpark du topic Kafka `wikistreams`, alimenté par
`wikistream_producer.py` avec les modifications récentes de Wikipédia.

1. **Démo** : nombre de modifications par robot, fenêtre d'une heure.
2. **Exploration 1, fenêtres fixes (tumbling)** : activité par wiki et par
   type d'auteur, minute par minute, avec la taille des modifications.
3. **Exploration 2, fenêtres glissantes (overlapping)** : fenêtres de
   10 minutes toutes les 2 minutes, contributeurs actifs et part des robots.
4. **Exploration 3, fenêtres glissantes** : pages les plus modifiées.
5. **Alerte sans état** : grosses modifications, ligne par ligne.

Les requêtes `console` écrivent dans les logs du conteneur
(`docker logs -f pyspark_notebook`). Les requêtes `memory` créent des tables
interrogeables en SQL pour afficher les résultats dans ce notebook.

In [1]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime

import pyspark
from pyspark.sql import functions as F

In [2]:
# initialize the spark session
# The Kafka connector must match the Spark version of the image.
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages",
            f"org.apache.spark:spark-sql-kafka-0-10_2.13:{pyspark.__version__}") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

Spark 4.2.0


In [3]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [4]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [5]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [6]:
# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .queryName("demo_bot_edits") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

## Enrichissement

- `event_time` : heure de l'événement côté Wikipédia (*event time*),
  convertie depuis les secondes Unix avec `timestamp_seconds`.
- `size_delta` : octets ajoutés (positif) ou retirés (négatif) par la
  modification ; `abs_size` est sa valeur absolue.
- Le *watermark* de 10 minutes indique à Spark combien de retard accepter
  avant de considérer une fenêtre comme terminée et d'oublier son état.

In [7]:
enriched_df = parsed_df \
    .withColumn("event_time", F.timestamp_seconds("timestamp")) \
    .withColumn("size_delta", col("length.new") - col("length.old")) \
    .withColumn("abs_size", F.abs(col("size_delta"))) \
    .withWatermark("event_time", "10 minutes")


def start_query(df, name, mode="complete"):
    """Start a console sink for the logs and a memory sink for the notebook."""
    df.writeStream.queryName(f"{name}_console").format("console") \
        .outputMode("update" if mode == "complete" else mode) \
        .option("truncate", False).option("numRows", 50).start()
    return df.writeStream.queryName(name).format("memory") \
        .outputMode(mode).start()

## Exploration 1 : fenêtres fixes (tumbling) d'une minute

`window(event_time, "1 minute")` découpe le temps en intervalles d'une minute
sans chevauchement : chaque modification appartient à **une seule** fenêtre.
Pour chaque wiki et type d'auteur (robot ou humain), on mesure le nombre de
modifications, les modifications mineures, les octets ajoutés et retirés et
la taille moyenne d'une modification.

In [8]:
tumbling_df = enriched_df \
    .groupBy(window("event_time", "1 minute"), "server_name", "bot") \
    .agg(
        count("*").alias("edits"),
        F.sum(col("minor").cast("int")).alias("minor_edits"),
        F.sum(F.when(col("size_delta") > 0, col("size_delta")).otherwise(0))
         .alias("bytes_added"),
        F.sum(F.when(col("size_delta") < 0, -col("size_delta")).otherwise(0))
         .alias("bytes_removed"),
        F.round(F.avg("abs_size"), 1).alias("avg_abs_size"),
    ) \
    .select(col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            "server_name", "bot", "edits", "minor_edits",
            "bytes_added", "bytes_removed", "avg_abs_size")

tumbling_query = start_query(tumbling_df, "tumbling_1min")

## Exploration 2 : fenêtres glissantes (overlapping) de 10 minutes

`window(event_time, "10 minutes", "2 minutes")` crée une fenêtre de
10 minutes qui démarre toutes les 2 minutes : les fenêtres se chevauchent et
chaque modification est comptée dans **cinq** fenêtres. Cela lisse l'activité
comme une moyenne mobile. On mesure par wiki le nombre de modifications, de
contributeurs distincts (approximation), la part des robots et la variation
nette de taille.

In [9]:
sliding_df = enriched_df \
    .groupBy(window("event_time", "10 minutes", "2 minutes"), "server_name") \
    .agg(
        count("*").alias("edits"),
        F.approx_count_distinct("user").alias("active_users"),
        F.round(F.avg(col("bot").cast("int")) * 100, 1).alias("bot_pct"),
        F.sum("size_delta").alias("net_bytes"),
        F.max("abs_size").alias("largest_edit"),
    ) \
    .select(col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            "server_name", "edits", "active_users", "bot_pct",
            "net_bytes", "largest_edit")

sliding_query = start_query(sliding_df, "sliding_10min")

## Exploration 3 : pages les plus modifiées, fenêtres glissantes de 5 minutes

Fenêtres de 5 minutes avançant d'une minute, comptées par page. Le
classement est fait ensuite en SQL sur la table mémoire, car `limit` et le
tri hors mode `complete` ne sont pas supportés sur un flux.

In [10]:
top_pages_df = enriched_df \
    .groupBy(window("event_time", "5 minutes", "1 minute"), "server_name", "title") \
    .agg(count("*").alias("edits"),
         F.approx_count_distinct("user").alias("users"),
         F.sum("size_delta").alias("net_bytes")) \
    .select(col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            "server_name", "title", "edits", "users", "net_bytes")

top_pages_query = top_pages_df.writeStream.queryName("top_pages") \
    .format("memory").outputMode("complete").start()

## Alerte sans état : grosses modifications

Un simple filtre, sans agrégation ni fenêtre, en mode `append` : chaque
modification de plus de 5 000 octets est écrite dans les logs dès qu'elle
arrive.

In [11]:
large_edits_query = enriched_df \
    .filter(col("abs_size") >= 5000) \
    .select("event_time", "server_name", "title", "user", "bot", "size_delta") \
    .writeStream.queryName("large_edits").format("console") \
    .outputMode("append").option("truncate", False).start()

## Résultats

`processAllAvailable()` attend que chaque requête ait traité tous les
messages déjà présents dans Kafka. Les requêtes continuent ensuite à tourner
et à écrire dans les logs ; relancer cette cellule rafraîchit les tableaux.

In [12]:
for query in spark.streams.active:
    query.processAllAvailable()
print("Active queries:", sorted(q.name for q in spark.streams.active))

spark.sql("""
SELECT server_name, COUNT(DISTINCT window_start) AS minutes,
       SUM(edits) AS edits,
       SUM(bytes_added) AS bytes_added, SUM(bytes_removed) AS bytes_removed
FROM tumbling_1min GROUP BY server_name ORDER BY edits DESC
""").show(truncate=False)

Active queries: ['demo_bot_edits', 'large_edits', 'sliding_10min', 'sliding_10min_console', 'top_pages', 'tumbling_1min', 'tumbling_1min_console']


+----------------+-------+-----+-----------+-------------+
|server_name     |minutes|edits|bytes_added|bytes_removed|
+----------------+-------+-----+-----------+-------------+
|en.wikipedia.org|13     |839  |168836     |84281        |
|fr.wikipedia.org|13     |313  |61019      |23556        |
|de.wikipedia.org|13     |248  |43756      |4933         |
|es.wikipedia.org|12     |113  |51137      |903          |
+----------------+-------+-----+-----------+-------------+



In [13]:
# Tumbling windows: one row per minute, wiki and author type
spark.sql("""
SELECT * FROM tumbling_1min
WHERE server_name = 'fr.wikipedia.org'
ORDER BY window_start DESC, bot LIMIT 20
""").show(truncate=False)

+-------------------+-------------------+----------------+-----+-----+-----------+-----------+-------------+------------+
|window_start       |window_end         |server_name     |bot  |edits|minor_edits|bytes_added|bytes_removed|avg_abs_size|
+-------------------+-------------------+----------------+-----+-----+-----------+-----------+-------------+------------+
|2026-10-16 09:25:00|2026-10-16 09:26:00|fr.wikipedia.org|false|11   |2          |732        |2981         |337.5       |
|2026-10-16 09:25:00|2026-10-16 09:26:00|fr.wikipedia.org|true |1    |0          |22         |0            |22.0        |
|2026-10-16 09:24:00|2026-10-16 09:25:00|fr.wikipedia.org|false|19   |5          |8457       |51           |447.8       |
|2026-10-16 09:24:00|2026-10-16 09:25:00|fr.wikipedia.org|true |2    |0          |26         |0            |13.0        |
|2026-10-16 09:23:00|2026-10-16 09:24:00|fr.wikipedia.org|false|20   |2          |8402       |214          |430.8       |
|2026-10-16 09:23:00|202

In [14]:
# Sliding windows: windows overlap, so the same event appears in several rows
spark.sql("""
SELECT * FROM sliding_10min
ORDER BY server_name, window_start DESC
""").show(40, truncate=False)

+-------------------+-------------------+----------------+-----+------------+-------+---------+------------+
|window_start       |window_end         |server_name     |edits|active_users|bot_pct|net_bytes|largest_edit|
+-------------------+-------------------+----------------+-----+------------+-------+---------+------------+
|2026-10-16 09:24:00|2026-10-16 09:34:00|de.wikipedia.org|30   |26          |16.7   |7731     |4506        |
|2026-10-16 09:22:00|2026-10-16 09:32:00|de.wikipedia.org|73   |53          |12.3   |13362    |4506        |
|2026-10-16 09:20:00|2026-10-16 09:30:00|de.wikipedia.org|119  |75          |11.8   |17420    |4506        |
|2026-10-16 09:18:00|2026-10-16 09:28:00|de.wikipedia.org|154  |88          |14.3   |26798    |4506        |
|2026-10-16 09:16:00|2026-10-16 09:26:00|de.wikipedia.org|194  |105         |13.9   |27967    |4506        |
|2026-10-16 09:14:00|2026-10-16 09:24:00|de.wikipedia.org|210  |115         |14.3   |29306    |4037        |
|2026-10-16 09:12:0

In [15]:
# Top 3 pages of each wiki over its last 5 minutes. The latest window ends
# 4 minutes after the last event (it only holds the last minute), so the
# window covering the last 5 minutes ends 4 minutes before it.
spark.sql("""
WITH latest AS (
  SELECT server_name, MAX(window_end) - INTERVAL 4 MINUTES AS window_end
  FROM top_pages GROUP BY server_name
), ranked AS (
  SELECT t.*, ROW_NUMBER() OVER (
           PARTITION BY t.server_name ORDER BY t.edits DESC, t.users DESC
         ) AS rank
  FROM top_pages t JOIN latest l
    ON t.server_name = l.server_name AND t.window_end = l.window_end
)
SELECT server_name, rank, window_start, window_end, title, edits, users,
       net_bytes
FROM ranked WHERE rank <= 3 ORDER BY server_name, rank
""").show(truncate=False)

+----------------+----+-------------------+-------------------+-------------------------------------------+-----+-----+---------+
|server_name     |rank|window_start       |window_end         |title                                      |edits|users|net_bytes|
+----------------+----+-------------------+-------------------+-------------------------------------------+-----+-----+---------+
|de.wikipedia.org|1   |2026-10-16 09:21:00|2026-10-16 09:26:00|Pressefoto des Jahres                      |3    |1    |369      |
|de.wikipedia.org|2   |2026-10-16 09:21:00|2026-10-16 09:26:00|Simone Sombecki                            |3    |1    |145      |
|de.wikipedia.org|3   |2026-10-16 09:21:00|2026-10-16 09:26:00|Wikipedia:Löschkandidaten/15. Oktober 2026 |2    |2    |597      |
|en.wikipedia.org|1   |2026-10-16 09:21:00|2026-10-16 09:26:00|User:Bunnypranav/Reports                   |6    |1    |-28      |
|en.wikipedia.org|2   |2026-10-16 09:21:00|2026-10-16 09:26:00|2026 Silver Rail Elementary

## Arrêt

Les requêtes tournent jusqu'à l'arrêt du kernel. Pour les arrêter à la main :

In [16]:
# for query in spark.streams.active:
#     query.stop()